# NYC Housing Price Prediction - Linear Regression from Scratch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Observe Dataset

In [ ]:
df = pd.read_csv('housing.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Nulls and Duplicates

In [ ]:
print('Null values:')
print(df.isnull().sum())
print('\nTotal nulls:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

null_counts = df.isnull().sum()
axes[0].bar(range(len(null_counts)), null_counts.values, color='steelblue')
axes[0].set_xticks(range(len(null_counts)))
axes[0].set_xticklabels(null_counts.index, rotation=45, ha='right', fontsize=8)
axes[0].set_title('Null Value Counts')

sample = df.isnull().astype(int).head(200)
axes[1].imshow(sample.T, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')
axes[1].set_yticks(range(len(sample.columns)))
axes[1].set_yticklabels(sample.columns, fontsize=7)
axes[1].set_title('Missing Data Heatmap')

plt.tight_layout()
plt.show()

## 3. Pre-processing

In [ ]:
df_clean = df.copy()

df_clean = df_clean.drop_duplicates()
print('After removing duplicates:', df_clean.shape)

df_clean = df_clean[df_clean['sale_price'] > 0]
df_clean = df_clean.dropna(subset=['sale_price'])
print('After removing zero prices:', df_clean.shape)

p1 = df_clean['sale_price'].quantile(0.01)
p99 = df_clean['sale_price'].quantile(0.99)
df_clean = df_clean[(df_clean['sale_price'] >= p1) & (df_clean['sale_price'] <= p99)]
print('After removing outliers:', df_clean.shape)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print('Remaining nulls:', df_clean.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['sale_price'] > 0]['sale_price'], bins=50, color='tomato', alpha=0.8)
axes[0].set_title('Sale Price - Raw')
axes[0].set_xlabel('Sale Price ($)')

axes[1].hist(df_clean['sale_price'], bins=50, color='teal', alpha=0.8)
axes[1].set_title('Sale Price - Cleaned')
axes[1].set_xlabel('Sale Price ($)')

plt.tight_layout()
plt.show()

## 4. Feature Correlation and Selection

In [ ]:
numeric_df = df_clean.select_dtypes(include=[np.number])
corr_with_price = numeric_df.corr()['sale_price'].drop('sale_price').sort_values(ascending=False)
print('Correlation with sale_price:')
print(corr_with_price)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['seagreen' if x > 0 else 'tomato' for x in corr_with_price.values]
axes[0].barh(range(len(corr_with_price)), corr_with_price.values, color=colors)
axes[0].set_yticks(range(len(corr_with_price)))
axes[0].set_yticklabels(corr_with_price.index, fontsize=9)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Feature Correlation with Sale Price')

corr_matrix = numeric_df.corr()
im = axes[1].imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr_matrix.columns)))
axes[1].set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=7)
axes[1].set_yticks(range(len(corr_matrix.columns)))
axes[1].set_yticklabels(corr_matrix.columns, fontsize=7)
axes[1].set_title('Correlation Matrix')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
features = ['bldgarea', 'lotarea', 'unitsres', 'numfloors']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df_clean[feat], df_clean['sale_price'], alpha=0.3, s=8)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Sale Price')
    axes[i].set_title(f'{feat} vs Sale Price')
    z = np.polyfit(df_clean[feat], df_clean['sale_price'], 1)
    x_line = np.linspace(df_clean[feat].min(), df_clean[feat].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), 'r-', linewidth=2)

plt.tight_layout()
plt.show()

## 5. Linear Regression from Scratch

In [ ]:
class LinearRegressionScratch:
    def __init__(self):
        self.theta = None
        self.mean_ = None
        self.std_ = None

    def fit(self, X, y):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        X_norm = (X - self.mean_) / (self.std_ + 1e-8)
        X_b = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])
        self.theta = np.linalg.lstsq(X_b.T @ X_b, X_b.T @ y, rcond=None)[0]

    def predict(self, X):
        X_norm = (X - self.mean_) / (self.std_ + 1e-8)
        X_b = np.hstack([np.ones((X_norm.shape[0], 1)), X_norm])
        return X_b @ self.theta

    def r2(self, y_true, y_pred):
        return 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)

    def rmse(self, y_true, y_pred):
        return np.sqrt(np.mean((y_true - y_pred)**2))

    def mae(self, y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))

In [ ]:
df_model = df_clean[features + ['sale_price']].dropna()
X = df_model[features].values.astype(float)
y = df_model['sale_price'].values.astype(float)

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

In [ ]:
model = LinearRegressionScratch()
model.fit(X_train, y_train)

print('Intercept:', round(model.theta[0], 2))
for feat, coef in zip(features, model.theta[1:]):
    print(f'{feat}: {round(coef, 2)}')

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print('Train R2:', round(model.r2(y_train, y_train_pred), 4))
print('Test  R2:', round(model.r2(y_test, y_test_pred), 4))
print('Train RMSE:', round(model.rmse(y_train, y_train_pred), 2))
print('Test  RMSE:', round(model.rmse(y_test, y_test_pred), 2))
print('Train MAE:', round(model.mae(y_train, y_train_pred), 2))
print('Test  MAE:', round(model.mae(y_test, y_test_pred), 2))

## 6. Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(y_test, y_test_pred, alpha=0.35, s=10)
mn, mx = min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())
axes[0,0].plot([mn, mx], [mn, mx], 'r--')
axes[0,0].set_xlabel('Actual Price')
axes[0,0].set_ylabel('Predicted Price')
axes[0,0].set_title('Actual vs Predicted')

residuals = y_test - y_test_pred
axes[0,1].scatter(y_test_pred, residuals, alpha=0.35, s=10, color='darkorange')
axes[0,1].axhline(0, color='red', linestyle='--')
axes[0,1].set_xlabel('Predicted Price')
axes[0,1].set_ylabel('Residuals')
axes[0,1].set_title('Residuals vs Predicted')

axes[1,0].hist(residuals, bins=50, color='teal', alpha=0.85)
axes[1,0].axvline(0, color='red', linestyle='--')
axes[1,0].set_title('Residual Distribution')

axes[1,1].barh(features, model.theta[1:])
axes[1,1].axvline(0, color='black', linewidth=0.8)
axes[1,1].set_title('Feature Coefficients')

plt.tight_layout()
plt.show()